# Phase 3 — Corruption Robustness Benchmark (Colab T4)

## Overview
Evaluates the baseline RT-DETR model against **25 corruption scenarios** (5 types × 5 severities),
following the ImageNet-C methodology. This phase is **pure measurement** — record results as-is.

### Corruption types
| Type | Description |
|---|---|
| `gaussian_noise` | Additive Gaussian pixel noise |
| `blur` | Gaussian blur (kernel size scales with severity) |
| `jpeg_compression` | JPEG re-encoding at decreasing quality |
| `brightness` | Global brightness reduction |
| `occlusion` | Random black rectangles masking scene content |

### Severity → parameter mapping
| Severity | noise σ | blur k | JPEG q | brightness | occlusion (n, frac) |
|---|---|---|---|---|---|
| 1 | 5 | 3 | 75 | 0.75 | 1 × 5% |
| 2 | 15 | 5 | 50 | 0.55 | 2 × 8% |
| 3 | 30 | 7 | 30 | 0.40 | 3 × 10% |
| 4 | 50 | 11 | 15 | 0.25 | 4 × 12% |
| 5 | 75 | 15 | 5 | 0.10 | 5 × 15% |

### Pre-requisites
- Phase 2 complete: `models/baseline/best.pt` exists in Drive
- Test split present: `data/annotated/images/test/` (61 images), labels at `data/annotated/labels/test/`

### Outputs
- `results/robustness_baseline.json` — all 25 mAP values + clean baseline + mPC
- `results/figures/robustness_curve_baseline.png` — line plot per corruption type
- `results/figures/viz3a_corruption_strip.png` — 5×5 heatmap

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────────
# Mount Drive, install deps, set all paths.

from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', '-q', 'ultralytics', 'imgaug'], check=True)

import os, json, shutil, tempfile, io
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
PROJECT_DIR  = '/content/drive/MyDrive/robot-perception'
WEIGHTS      = f'{PROJECT_DIR}/models/baseline/best.pt'
TEST_IMG_DIR = f'{PROJECT_DIR}/data/annotated/images/test'
TEST_LBL_DIR = f'{PROJECT_DIR}/data/annotated/labels/test'
RESULTS_DIR  = f'{PROJECT_DIR}/results/figures'
RESULTS_JSON = f'{PROJECT_DIR}/results/robustness_baseline.json'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────────────────
CLASS_NAMES  = ['arm', 'leg', 'torso', 'head']   # 4 classes, IDs 0-3
CORRUPTIONS  = ['gaussian_noise', 'blur', 'jpeg_compression', 'brightness', 'occlusion']
SEVERITIES   = [1, 2, 3, 4, 5]

# Verify pre-requisites
assert os.path.exists(WEIGHTS),      f'Weights not found: {WEIGHTS}\nRun Phase 2 first.'
test_images = sorted(
    str(p) for p in Path(TEST_IMG_DIR).glob('*')
    if p.suffix.lower() in ('.jpg', '.jpeg', '.png')
)
assert len(test_images) > 0, f'No test images found at {TEST_IMG_DIR}'
print(f'Weights  : {WEIGHTS}')
print(f'Test images: {len(test_images)}')
print(f'Corruptions: {CORRUPTIONS}')
print(f'Total eval runs: {len(CORRUPTIONS) * len(SEVERITIES)} + 1 clean = 26')

In [ ]:
# ── Cell 2: Corruption functions ───────────────────────────────────────────────
# apply_corruption(image_rgb, corruption_type, severity) -> corrupted uint8 RGB array
#
# Severity → parameter mapping (authoritative):
#
#   gaussian_noise : sigma  = [  5,  15,  30,  50,  75 ][s-1]
#   blur           : kernel = [  3,   5,   7,  11,  15 ][s-1]  (must be odd)
#   jpeg_compression: quality=[75,  50,  30,  15,   5 ][s-1]
#   brightness     : factor = [0.75,0.55,0.40,0.25,0.10][s-1]
#   occlusion      : (n_boxes, box_frac) per severity:
#                    s1=(1,0.05)  s2=(2,0.08)  s3=(3,0.10)  s4=(4,0.12)  s5=(5,0.15)

def apply_corruption(image: np.ndarray, corruption_type: str, severity: int) -> np.ndarray:
    """
    Apply a single corruption at a given severity to an RGB uint8 image.
    Returns a new RGB uint8 array; input is not mutated.
    """
    assert 1 <= severity <= 5, f'severity must be 1-5, got {severity}'
    s = severity - 1  # 0-indexed
    img = image.copy()

    # ── 1. Gaussian noise ──────────────────────────────────────────────────────
    if corruption_type == 'gaussian_noise':
        sigma = [5, 15, 30, 50, 75][s]
        noise = np.random.normal(0, sigma, img.shape).astype(np.float32)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    # ── 2. Gaussian blur ───────────────────────────────────────────────────────
    elif corruption_type == 'blur':
        kernel = [3, 5, 7, 11, 15][s]
        img = cv2.GaussianBlur(img, (kernel, kernel), 0)

    # ── 3. JPEG compression ────────────────────────────────────────────────────
    elif corruption_type == 'jpeg_compression':
        quality = [75, 50, 30, 15, 5][s]
        pil_img = Image.fromarray(img)
        buf = io.BytesIO()
        pil_img.save(buf, format='JPEG', quality=quality)
        buf.seek(0)
        img = np.array(Image.open(buf))

    # ── 4. Brightness reduction ────────────────────────────────────────────────
    elif corruption_type == 'brightness':
        factor = [0.75, 0.55, 0.40, 0.25, 0.10][s]
        img = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)

    # ── 5. Occlusion (random black rectangles) ─────────────────────────────────
    elif corruption_type == 'occlusion':
        params = [(1, 0.05), (2, 0.08), (3, 0.10), (4, 0.12), (5, 0.15)][s]
        n_boxes, box_frac = params
        h, w = img.shape[:2]
        box_h = int(h * box_frac)
        box_w = int(w * box_frac)
        rng = np.random.default_rng(seed=42)  # fixed seed → deterministic benchmark
        for _ in range(n_boxes):
            x = int(rng.integers(0, max(1, w - box_w)))
            y = int(rng.integers(0, max(1, h - box_h)))
            img[y:y + box_h, x:x + box_w] = 0

    else:
        raise ValueError(f'Unknown corruption type: {corruption_type}')

    return img


# ── Quick smoke-test ────────────────────────────────────────────────────────────
_dummy = np.random.randint(0, 255, (100, 100, 3), dtype=np.uint8)
for _c in CORRUPTIONS:
    for _s in SEVERITIES:
        _out = apply_corruption(_dummy, _c, _s)
        assert _out.shape == _dummy.shape, f'Shape mismatch for {_c}/s{_s}'
print('Corruption functions OK — all 25 combinations smoke-tested.')

In [ ]:
# ── Cell 3: Generate & Evaluate all 25 corruption combinations ─────────────────
#
# Strategy: corrupted images are written to a temp dir on local /content (fast SSD),
# evaluated, then the temp dir is deleted after each corruption×severity combo.
# Nothing is permanently saved to disk — Drive bandwidth is preserved.
#
# Ultralytics model.val() requires:
#   - a YAML pointing at a directory whose images/ sub-folder holds images
#   - labels in an adjacent labels/ sub-folder (same stem, .txt extension)
# We build that layout in a fresh tempdir per combo, run val, then shutil.rmtree it.

import tempfile, shutil, glob
from ultralytics import RTDETR

# Load model once
model = RTDETR(WEIGHTS)

# ── Helper: write YAML for a temp directory ────────────────────────────────────
def _write_yaml(root_dir: str, nc: int, names: list) -> str:
    yaml_path = os.path.join(root_dir, 'dataset.yaml')
    content = (
        f'path: {root_dir}\n'
        f'train: images\n'
        f'val: images\n'
        f'test: images\n'
        f'nc: {nc}\n'
        f"names: {names}\n"
    )
    with open(yaml_path, 'w') as f:
        f.write(content)
    return yaml_path

# ── Helper: evaluate model on a set of (image_array, label_path) pairs ─────────
def evaluate_on_images(model, image_arrays, label_paths, nc, names):
    """
    Write images + labels to a temp dir, run model.val(), return mAP@0.5.
    Cleans up the temp dir on exit.
    """
    tmp = tempfile.mkdtemp(dir='/content')
    img_dir = os.path.join(tmp, 'images')
    lbl_dir = os.path.join(tmp, 'labels')
    os.makedirs(img_dir)
    os.makedirs(lbl_dir)

    for i, (arr, lbl_src) in enumerate(zip(image_arrays, label_paths)):
        stem = Path(lbl_src).stem
        # Write corrupted image as JPEG
        img_bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
        cv2.imwrite(os.path.join(img_dir, f'{stem}.jpg'), img_bgr)
        # Copy label
        if os.path.exists(lbl_src):
            shutil.copy2(lbl_src, os.path.join(lbl_dir, f'{stem}.txt'))

    yaml_path = _write_yaml(tmp, nc, names)

    # Clear any stale Ultralytics caches
    for cache_file in glob.glob(os.path.join(lbl_dir, '*.cache')):
        os.remove(cache_file)

    try:
        metrics = model.val(data=yaml_path, split='test', verbose=False)
        mAP = float(metrics.box.map50)
    except Exception as e:
        print(f'    WARNING: val() raised exception: {e}')
        mAP = 0.0
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

    return mAP

# ── Pre-load all test images into RAM as RGB arrays (61 images, ~few MB) ────────
print('Loading test images into memory...')
clean_arrays = []
label_paths  = []
for img_path in test_images:
    stem = Path(img_path).stem
    bgr  = cv2.imread(img_path)
    if bgr is None:
        print(f'  WARNING: could not read {img_path}, skipping')
        continue
    clean_arrays.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    label_paths.append(os.path.join(TEST_LBL_DIR, f'{stem}.txt'))
print(f'Loaded {len(clean_arrays)} images.')

# ── Evaluate clean baseline ────────────────────────────────────────────────────
print('\nEvaluating clean baseline...')
clean_mAP = evaluate_on_images(model, clean_arrays, label_paths, len(CLASS_NAMES), CLASS_NAMES)
print(f'Clean mAP@0.5: {clean_mAP:.4f}')

# ── Evaluate all 25 corruption × severity combinations ────────────────────────
results_grid = {}  # results_grid[corruption_type][severity_int] = mAP float
total = len(CORRUPTIONS) * len(SEVERITIES)
done  = 0

for corruption in CORRUPTIONS:
    results_grid[corruption] = {}
    for severity in SEVERITIES:
        corrupted_arrays = [
            apply_corruption(arr, corruption, severity)
            for arr in clean_arrays
        ]
        mAP = evaluate_on_images(
            model, corrupted_arrays, label_paths, len(CLASS_NAMES), CLASS_NAMES
        )
        results_grid[corruption][severity] = mAP
        done += 1
        print(f'  [{done:2d}/{total}] {corruption}/s{severity}: mAP@0.5 = {mAP:.4f}')

print('\nAll 25 corruption sets evaluated.')

In [ ]:
# ── Cell 4: Compute mPC and save results/robustness_baseline.json ──────────────
#
# mPC = mean Performance under Corruption = arithmetic mean of all 25 mAP values.
# relative_mPC = mPC / clean_mAP  (fraction of clean performance retained).

all_mAPs     = [results_grid[c][s] for c in CORRUPTIONS for s in SEVERITIES]
mPC          = float(np.mean(all_mAPs))
relative_mPC = mPC / clean_mAP if clean_mAP > 0 else 0.0

# Per-corruption averages (useful for diagnosing worst corruption type)
per_corruption_avg = {c: float(np.mean([results_grid[c][s] for s in SEVERITIES]))
                      for c in CORRUPTIONS}
worst_corruption   = min(per_corruption_avg, key=per_corruption_avg.get)
avg_s5             = float(np.mean([results_grid[c][5] for c in CORRUPTIONS]))

output = {
    'weights'       : WEIGHTS,
    'n_test_images' : len(clean_arrays),
    'clean_mAP50'   : clean_mAP,
    'mPC'           : mPC,
    'relative_mPC'  : relative_mPC,
    'per_corruption_avg': per_corruption_avg,
    'avg_mAP_severity5' : avg_s5,
    # Store severity keys as strings (JSON requires string keys)
    'results_grid'  : {
        c: {str(s): results_grid[c][s] for s in SEVERITIES}
        for c in CORRUPTIONS
    },
}

with open(RESULTS_JSON, 'w') as f:
    json.dump(output, f, indent=2)

print('=== Phase 3 — Robustness Summary (baseline) ===')
print(f'  Test images     : {len(clean_arrays)}')
print(f'  Clean mAP@0.5   : {clean_mAP:.4f}')
print(f'  mPC             : {mPC:.4f}')
print(f'  Relative mPC    : {relative_mPC:.4f}  ({relative_mPC * 100:.1f}% of clean retained)')
print(f'  Worst corruption: {worst_corruption}  (avg mAP={per_corruption_avg[worst_corruption]:.4f})')
print(f'  Avg mAP at s=5  : {avg_s5:.4f}')
print(f'\nSaved: {RESULTS_JSON}')

if relative_mPC >= 0.85:
    print('\nWARNING: Relative mPC >= 0.85 — model appears suspiciously robust.')
    print('  Double-check apply_corruption() to ensure corruptions are actually applied.')

In [ ]:
# ── Cell 5: VIZ — Robustness curve ────────────────────────────────────────────
# Line plot: x = severity (1-5), y = mAP@0.5, one line per corruption type.
# Clean baseline shown as a horizontal dashed line.
# Saved to: results/figures/robustness_curve_baseline.png

COLORS = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00', '#984ea3']

fig, ax = plt.subplots(figsize=(9, 5))

for i, corruption in enumerate(CORRUPTIONS):
    mAPs = [results_grid[corruption][s] for s in SEVERITIES]
    ax.plot(SEVERITIES, mAPs, marker='o', linewidth=2,
            color=COLORS[i], label=corruption)

ax.axhline(y=clean_mAP, color='black', linestyle='--', linewidth=1.5,
           label=f'clean baseline ({clean_mAP:.3f})')

ax.set_xlabel('Corruption Severity', fontsize=12)
ax.set_ylabel('mAP@0.5', fontsize=12)
ax.set_title('Robustness Curve — Baseline RT-DETR', fontsize=13)
ax.set_xticks(SEVERITIES)
ax.set_xlim(0.8, 5.2)
ax.set_ylim(bottom=0)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)

out_path = f'{RESULTS_DIR}/robustness_curve_baseline.png'
plt.tight_layout()
plt.savefig(out_path, dpi=150)
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# ── Cell 6: VIZ — 5×5 mAP heatmap (corruption type × severity) ────────────────
# Rows = corruption type (5), columns = severity 1-5.
# Cell colour = mAP@0.5 value.  Saved to: results/figures/viz3a_corruption_strip.png

grid_data = np.array([
    [results_grid[c][s] for s in SEVERITIES]
    for c in CORRUPTIONS
])  # shape (5, 5)

fig, ax = plt.subplots(figsize=(8, 4.5))
im = ax.imshow(grid_data, cmap='RdYlGn', vmin=0, vmax=clean_mAP, aspect='auto')

ax.set_xticks(range(5))
ax.set_xticklabels([f'Severity {s}' for s in SEVERITIES], fontsize=10)
ax.set_yticks(range(len(CORRUPTIONS)))
ax.set_yticklabels(CORRUPTIONS, fontsize=10)
ax.set_title('mAP@0.5 Heatmap — Baseline RT-DETR\n(green = near clean performance, red = severe drop)',
             fontsize=11)

# Annotate each cell with its mAP value
for i in range(len(CORRUPTIONS)):
    for j in range(len(SEVERITIES)):
        val = grid_data[i, j]
        text_color = 'black' if val > clean_mAP * 0.4 else 'white'
        ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                fontsize=9, color=text_color, fontweight='bold')

plt.colorbar(im, ax=ax, label='mAP@0.5')

out_path = f'{RESULTS_DIR}/viz3a_corruption_strip.png'
plt.tight_layout()
plt.savefig(out_path, dpi=150)
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# ── Cell 7: Metrics check — print full 25-entry table, confirm completeness ────

evaluated = sum(1 for c in CORRUPTIONS for s in SEVERITIES
                if results_grid.get(c, {}).get(s) is not None)

print(f'{"Corruption":<22} {"S1":>7} {"S2":>7} {"S3":>7} {"S4":>7} {"S5":>7} {"Avg":>7}')
print('-' * 65)
for c in CORRUPTIONS:
    row = [results_grid[c][s] for s in SEVERITIES]
    avg = np.mean(row)
    vals = '  '.join(f'{v:.4f}' for v in row)
    print(f'{c:<22} {vals}  {avg:.4f}')

print('-' * 65)
# Column averages (per severity)
for s in SEVERITIES:
    col_avg = np.mean([results_grid[c][s] for c in CORRUPTIONS])

col_avgs = [np.mean([results_grid[c][s] for c in CORRUPTIONS]) for s in SEVERITIES]
col_str  = '  '.join(f'{v:.4f}' for v in col_avgs)
print(f'{"Per-severity avg":<22} {col_str}  {mPC:.4f}')

print()
print(f'Total evaluated: {evaluated}/25')
assert evaluated == 25, f'INCOMPLETE — only {evaluated}/25 combinations evaluated!'
print('All 25 combinations confirmed.')
print(f'\nClean mAP@0.5 : {clean_mAP:.4f}')
print(f'mPC           : {mPC:.4f}')
print(f'Relative mPC  : {relative_mPC:.4f}')

# Phase 3 — Completion Checklist

Before marking Phase 3 done, confirm every item below.

## Metrics
- [ ] All 25 corruption × severity combinations evaluated (Cell 7 must print "All 25 combinations confirmed")
- [ ] `clean_mAP@0.5` recorded
- [ ] `mPC` computed and recorded
- [ ] `relative_mPC` recorded

## Files saved
- [ ] `results/robustness_baseline.json` exists and contains `results_grid` with all 25 entries
- [ ] `results/figures/robustness_curve_baseline.png` saved (line plot, 5 lines + dashed clean baseline)
- [ ] `results/figures/viz3a_corruption_strip.png` saved (5×5 heatmap with mAP values annotated)

## Sanity checks
- [ ] Robustness curve shows a downward trend for at least most corruption types as severity increases
- [ ] Severity 5 mAP values are lower than Severity 1 values for all corruption types (if not, re-check `apply_corruption`)
- [ ] If `relative_mPC >= 0.85`, investigate — corruptions may not be applying correctly
- [ ] During `model.val()` output, instance count must be non-zero. If zero, label path layout is broken.

## Next step
Proceed to `04_robust_training.ipynb`.

**Note:** the corrupted images generated in this notebook are ephemeral (temp dir on `/content`).
Phase 4 will re-apply corruptions to evaluate the robust model — no persistent storage needed.